In [ ]:
# torch, faiss (k-NN/perplexity graph construction) and PHATE's numba-based internals
# all get loaded into one process here; with unconstrained thread pools that combination
# makes macOS abort with a duplicate-OpenMP-runtime error (kernel crash / segfault).
# Capping thread counts before any of those libraries import, plus the (unsafe but
# standard) escape hatch as a second line of defense, avoids it.
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path


project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))


# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from plot_utilities import *
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import phate
import subprocess


# UMAP and Concord only exist in their own conda envs (the env running this notebook has
# phate but not umap/anndata/concord), so both are invoked as subprocesses: each reads
# <buffer_id>.npy from its workdir and writes res_<buffer_id>.npy there. The buffer id is
# per-process so two concurrent runs cannot race on the same filenames.
def external_embed(env, workdir, script, buffer_id, X):
    np.save(f'{workdir}/{buffer_id}.npy', X)
    subprocess.run(f'cd {workdir} && conda run -n {env} python {script} {buffer_id}', shell=True)
    Y = np.load(f'{workdir}/res_{buffer_id}.npy')
    os.remove(f'{workdir}/{buffer_id}.npy')
    os.remove(f'{workdir}/res_{buffer_id}.npy')
    return Y


def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config


# ERA5_Graph_Generator.generate_normed_sample() draws the sampled point subset with
# np.random.choice, and nothing in the sampling path re-seeds, so the sampled graph -- and
# every embedding, statistic and figure derived from it -- changes run-to-run unless the
# global RNGs are seeded first. The in-notebook methods pass random_state=42 explicitly and
# the UMAP/concord runner scripts hardcode their own seed, so seeding here is what makes
# the sampled input itself reproducible.
SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()


In [ ]:
model, config = load_model(id='regular_papermodel2_2026-05-04_t12-01-43' , model_type='regular')#test1_model5_median_tkernel_2026-04-22_t21-22-37


In [ ]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [ ]:
random.randint(100,1000)

In [ ]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)

bbox = lon_min, lat_min, lon_max, lat_max


print('generate data')

graphs=[]
locations_list=[]
X_list=[]

# ERA5_Graph_Generator.sample_graph draws with np.random.choice and the sample size
# comes from random.randint below, so seed the global RNGs first to make the whole run
# reproducible. set_seed() is the same helper the imports cell defines (random, numpy,
# torch at SEED); seeding once here keeps the per-date N sequence identical to before.
set_seed()

for date in dates:
    N=random.randint(1000,10000)
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
    graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
    sample = graph.to(device)
    graphs.append(sample)
    locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
    X_list.append(sample.normed_values.detach().cpu().numpy())

In [ ]:
pred_nofe_list = []
pred_pca_list = []
pred_tsne_list = []
pred_umap_list = []
pred_phate_list = []
pred_spatialpca_list = []
pred_concord_list = []

# Start the spatialPCA R process ONCE, outside the loop -- library(SpatialPCA) (Seurat,
# SPARK, etc.) is the slow part to load, and conda run's own activation overhead also
# used to get paid on every single call. It now stays alive and answers one request per
# line over stdin/stdout for the whole loop below.
spdir = f'{project_root}/ERA5/spatialpca'
# Unique per process so concurrently running scripts/notebooks don't collide on the
# same buffer filenames (a fixed call_id caused FileNotFoundError / silent data
# corruption when two spatialPCA calls raced on the same X.csv/coords.csv paths).
spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'
spatialpca_proc = subprocess.Popen(
    ['conda', 'run', '-n', 'spatialpca', '--no-capture-output', 'Rscript', 'execute_spatialpca_server.R'],
    cwd=spdir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, bufsize=1,
    # R searches the *user* library (~/Library/R/<arch>/<ver>/library) before the conda
    # env's own library. CRAN-built packages there link the CRAN R framework rather than
    # conda's libR, and SpatialPCA imports Rcpp -- whose foreign .so then segfaults at
    # dyn.load. Pointing R_LIBS_USER at a nonexistent path lets the env's own packages win.
    env=dict(os.environ, R_LIBS_USER='/nonexistent'),
)
ready = spatialpca_proc.stdout.readline().strip()
if ready != 'READY':
    raise RuntimeError(f'spatialPCA server failed to start: {ready!r}\n{spatialpca_proc.stderr.read()}')
print('spatialPCA server ready.')


def spatialpca_embed(X, coords, call_id=spatialpca_call_id, n_components=3):
    np.savetxt(f'{spdir}/{call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{spdir}/{call_id}_coords.csv', coords, delimiter=',')

    spatialpca_proc.stdin.write(f'{call_id} {n_components}\n')
    spatialpca_proc.stdin.flush()
    response = spatialpca_proc.stdout.readline().strip()

    os.remove(f'{spdir}/{call_id}_X.csv')
    os.remove(f'{spdir}/{call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {call_id}: {response}')

    Y = np.loadtxt(f'{spdir}/res_{call_id}.csv', delimiter=',', skiprows=1)
    os.remove(f'{spdir}/res_{call_id}.csv')
    return Y

for sample in graphs:
    # NOFE
    print('NOFE ...', end='\r')
    pred_nofe = model(sample).detach().cpu().numpy()
    pred_nofe_list.append(pred_nofe)
    
    X = sample.normed_values.detach().cpu().numpy()
    
    # PCA
    print('PCA ...', end='\r')
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    pred_pca_list.append(pred_pca)

    # t-SNE
    print('t-SNE ...', end='\r')
    pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)
    pred_tsne_list.append(pred_tsne)

    # UMAP
    print('UMAP ...', end='\r')
    pred_umap = external_embed(
        'umap', f'{project_root}/ERA5/umap', 'execute_umap.py',
        f'umap_buffer_{os.getpid()}', X,
    )
    pred_umap_list.append(pred_umap)

    # PHATE
    print('PHATE ...', end='\r')
    pred_phate = phate.PHATE(
        n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
    ).fit_transform(X)
    pred_phate_list.append(pred_phate)

    # spatialPCA
    print('spatialPCA ...', end='\r')
    pred_spatialpca = spatialpca_embed(X, sample.normed_coords.detach().cpu().numpy())
    pred_spatialpca_list.append(pred_spatialpca)

    # Concord
    print('Concord ...', end='\r')
    pred_concord = external_embed(
        'concord', f'{project_root}/ERA5/concord', 'execute_concord.py',
        f'concord_buffer_{os.getpid()}', X,
    )
    pred_concord_list.append(pred_concord)

# Shut the spatialPCA server down (normal path only; on error the run already fails loudly
# and the server is a child of this kernel).
spatialpca_proc.stdin.write('QUIT\n')
spatialpca_proc.stdin.flush()
spatialpca_proc.wait(timeout=10)


In [ ]:
def pairwise_distances(X):
    # X: (N, d)
    norms = np.sum(X**2, axis=1, keepdims=True)  # (N, 1)
    
    dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
    
    # numerical stability: avoid tiny negatives
    dist2 = np.maximum(dist2, 0.0)
    
    return np.sqrt(dist2)

def distance_preservation(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    
    # Take upper triangle (avoid duplicates + diagonal)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    
    # Correlation (1 = perfect preservation)
    corr = np.corrcoef(dH, dL)[0, 1]
    
    return corr

def stress(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    
    # Take upper triangle (avoid duplicates + diagonal)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    
    # stress 
    stress = np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))
    
    return stress

def stress_scale_free(H, L):
    """Kruskal stress-1 after rescaling L by its least-squares factor.

    `stress` above compares raw distances, so it also charges an embedding for living on a
    different scale than H. t-SNE is the case in point: sklearn's TSNE(learning_rate='auto')
    converges to a globally ~10x larger layout on some dates, which alone inflates the raw
    stress of an otherwise mid-pack method. Dividing dL by the optimal factor
    sum(dH*dL)/sum(dL^2) removes that degree of freedom, so this measures shape only and is
    comparable across methods.
    """
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)

    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]

    scale = np.sum(dH * dL) / np.sum(dL**2)
    return np.sqrt(np.sum((dH - scale * dL)**2) / np.sum(dH**2))


In [ ]:

def edge_distances(X, edge_indices):
    # X: (N, d), edge_indices is torch_geometric's (2, E): row 0 sources, row 1 targets.
    diff = X[edge_indices[0]] - X[edge_indices[1]]
    return np.linalg.norm(diff, axis=1)

def edge_distance_preservation(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    # Correlation
    corr = np.corrcoef(dH, dL)[0, 1]
    return corr

def edge_stress(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    return np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))

def edge_stress_scale_free(H, L, edge_indices):
    """Scale-free counterpart of `edge_stress`: same least-squares rescaling of dL."""
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)

    scale = np.sum(dH * dL) / np.sum(dL**2)
    return np.sqrt(np.sum((dH - scale * dL)**2) / np.sum(dH**2))

#def weighted_edge_stress(H, L, edge_indices, weights):
#    dH = edge_distances(H, edge_indices)
#    dL = edge_distances(L, edge_indices)   
#    return np.sqrt(np.sum(weights * (dH - dL)**2) / np.sum(weights * dH**2))

In [ ]:
nofe_preservation = []
nofe_stress = []
nofe_edge_preservation = []
nofe_edge_stress = []
nofe_stress_scale_free = []
nofe_edge_stress_scale_free = []

pca_preservation = []
pca_stress = []
pca_edge_preservation = []
pca_edge_stress = []
pca_stress_scale_free = []
pca_edge_stress_scale_free = []

tsne_preservation = []
tsne_stress = []
tsne_edge_preservation = []
tsne_edge_stress = []
tsne_stress_scale_free = []
tsne_edge_stress_scale_free = []

umap_preservation = []
umap_stress = []
umap_edge_preservation = []
umap_edge_stress = []
umap_stress_scale_free = []
umap_edge_stress_scale_free = []

phate_preservation = []
phate_stress = []
phate_edge_preservation = []
phate_edge_stress = []
phate_stress_scale_free = []
phate_edge_stress_scale_free = []

spatialpca_preservation = []
spatialpca_stress = []
spatialpca_edge_preservation = []
spatialpca_edge_stress = []
spatialpca_stress_scale_free = []
spatialpca_edge_stress_scale_free = []
pca_stress_scale_free = []
pca_edge_stress_scale_free = []

concord_preservation = []
concord_stress = []
concord_edge_preservation = []
concord_edge_stress = []
concord_stress_scale_free = []
concord_edge_stress_scale_free = []

for i in range(len(graphs)):

    edge_indices = graphs[i].edge_index.detach().cpu().numpy()
    H = graphs[i].normed_values.detach().cpu().numpy()
    Lnofe = pred_nofe_list[i]
    Lpca = pred_pca_list[i]
    Ltsne = pred_tsne_list[i]
    Lumap = pred_umap_list[i]
    Lphate = pred_phate_list[i]
    Lspatialpca = pred_spatialpca_list[i]
    Lconcord = pred_concord_list[i]

    nofe_preservation.append(distance_preservation(H, Lnofe))
    nofe_stress.append(stress(H, Lnofe))
    nofe_edge_preservation.append(edge_distance_preservation(H, Lnofe, edge_indices))
    nofe_edge_stress.append(edge_stress(H, Lnofe, edge_indices))
    nofe_stress_scale_free.append(stress_scale_free(H, Lnofe))
    nofe_edge_stress_scale_free.append(edge_stress_scale_free(H, Lnofe, edge_indices))

    pca_preservation.append(distance_preservation(H, Lpca))
    pca_stress.append(stress(H, Lpca))
    pca_edge_preservation.append(edge_distance_preservation(H, Lpca, edge_indices))
    pca_edge_stress.append(edge_stress(H, Lpca, edge_indices))
    pca_stress_scale_free.append(stress_scale_free(H, Lpca))
    pca_edge_stress_scale_free.append(edge_stress_scale_free(H, Lpca, edge_indices))

    tsne_preservation.append(distance_preservation(H, Ltsne))
    tsne_stress.append(stress(H, Ltsne))
    tsne_edge_preservation.append(edge_distance_preservation(H, Ltsne, edge_indices))
    tsne_edge_stress.append(edge_stress(H, Ltsne, edge_indices))
    tsne_stress_scale_free.append(stress_scale_free(H, Ltsne))
    tsne_edge_stress_scale_free.append(edge_stress_scale_free(H, Ltsne, edge_indices))

    umap_preservation.append(distance_preservation(H, Lumap))
    umap_stress.append(stress(H, Lumap))
    umap_edge_preservation.append(edge_distance_preservation(H, Lumap, edge_indices))
    umap_edge_stress.append(edge_stress(H, Lumap, edge_indices))
    umap_stress_scale_free.append(stress_scale_free(H, Lumap))
    umap_edge_stress_scale_free.append(edge_stress_scale_free(H, Lumap, edge_indices))

    phate_preservation.append(distance_preservation(H, Lphate))
    phate_stress.append(stress(H, Lphate))
    phate_edge_preservation.append(edge_distance_preservation(H, Lphate, edge_indices))
    phate_edge_stress.append(edge_stress(H, Lphate, edge_indices))
    phate_stress_scale_free.append(stress_scale_free(H, Lphate))
    phate_edge_stress_scale_free.append(edge_stress_scale_free(H, Lphate, edge_indices))

    spatialpca_preservation.append(distance_preservation(H, Lspatialpca))
    spatialpca_stress.append(stress(H, Lspatialpca))
    spatialpca_edge_preservation.append(edge_distance_preservation(H, Lspatialpca, edge_indices))
    spatialpca_edge_stress.append(edge_stress(H, Lspatialpca, edge_indices))
    spatialpca_stress_scale_free.append(stress_scale_free(H, Lspatialpca))
    spatialpca_edge_stress_scale_free.append(edge_stress_scale_free(H, Lspatialpca, edge_indices))

    concord_preservation.append(distance_preservation(H, Lconcord))
    concord_stress.append(stress(H, Lconcord))
    concord_edge_preservation.append(edge_distance_preservation(H, Lconcord, edge_indices))
    concord_edge_stress.append(edge_stress(H, Lconcord, edge_indices))
    concord_stress_scale_free.append(stress_scale_free(H, Lconcord))
    concord_edge_stress_scale_free.append(edge_stress_scale_free(H, Lconcord, edge_indices))


score_cols = {
    'nofe': [np.mean(nofe_preservation), np.mean(nofe_stress), np.mean(nofe_edge_preservation), np.mean(nofe_edge_stress), np.mean(nofe_stress_scale_free), np.mean(nofe_edge_stress_scale_free)],
    'pca': [np.mean(pca_preservation), np.mean(pca_stress), np.mean(pca_edge_preservation), np.mean(pca_edge_stress), np.mean(pca_stress_scale_free), np.mean(pca_edge_stress_scale_free)],
    'tsne': [np.mean(tsne_preservation), np.mean(tsne_stress), np.mean(tsne_edge_preservation), np.mean(tsne_edge_stress), np.mean(tsne_stress_scale_free), np.mean(tsne_edge_stress_scale_free)],
    'umap': [np.mean(umap_preservation), np.mean(umap_stress), np.mean(umap_edge_preservation), np.mean(umap_edge_stress), np.mean(umap_stress_scale_free), np.mean(umap_edge_stress_scale_free)],
    'phate': [np.mean(phate_preservation), np.mean(phate_stress), np.mean(phate_edge_preservation), np.mean(phate_edge_stress), np.mean(phate_stress_scale_free), np.mean(phate_edge_stress_scale_free)],
    'spatialpca': [np.mean(spatialpca_preservation), np.mean(spatialpca_stress), np.mean(spatialpca_edge_preservation), np.mean(spatialpca_edge_stress), np.mean(spatialpca_stress_scale_free), np.mean(spatialpca_edge_stress_scale_free)],
    'concord': [np.mean(concord_preservation), np.mean(concord_stress), np.mean(concord_edge_preservation), np.mean(concord_edge_stress), np.mean(concord_stress_scale_free), np.mean(concord_edge_stress_scale_free)],
}
std_cols = {
    'nofe': [np.std(nofe_preservation), np.std(nofe_stress), np.std(nofe_edge_preservation), np.std(nofe_edge_stress), np.std(nofe_stress_scale_free), np.std(nofe_edge_stress_scale_free)],
    'pca': [np.std(pca_preservation), np.std(pca_stress), np.std(pca_edge_preservation), np.std(pca_edge_stress), np.std(pca_stress_scale_free), np.std(pca_edge_stress_scale_free)],
    'tsne': [np.std(tsne_preservation), np.std(tsne_stress), np.std(tsne_edge_preservation), np.std(tsne_edge_stress), np.std(tsne_stress_scale_free), np.std(tsne_edge_stress_scale_free)],
    'umap': [np.std(umap_preservation), np.std(umap_stress), np.std(umap_edge_preservation), np.std(umap_edge_stress), np.std(umap_stress_scale_free), np.std(umap_edge_stress_scale_free)],
    'phate': [np.std(phate_preservation), np.std(phate_stress), np.std(phate_edge_preservation), np.std(phate_edge_stress), np.std(phate_stress_scale_free), np.std(phate_edge_stress_scale_free)],
    'spatialpca': [np.std(spatialpca_preservation), np.std(spatialpca_stress), np.std(spatialpca_edge_preservation), np.std(spatialpca_edge_stress), np.std(spatialpca_stress_scale_free), np.std(spatialpca_edge_stress_scale_free)],
    'concord': [np.std(concord_preservation), np.std(concord_stress), np.std(concord_edge_preservation), np.std(concord_edge_stress), np.std(concord_stress_scale_free), np.std(concord_edge_stress_scale_free)],
}

scores = pd.DataFrame(score_cols, index=['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)',
       'stress (global, scale-free)', 'stress (local, scale-free)'])
#print(scores)
print(', '.join(score_cols.keys()))
for i, type in enumerate(['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)',
                            'stress (global, scale-free)', 'stress (local, scale-free)']):
    print(f'{type}: ' + ' & '.join(f'{score_cols[name][i]:.3f} $\\pm$ {std_cols[name][i]:.3f}' for name in score_cols))


In [ ]:
# Comprehensive table: all seven methods, computed the same way (correlation/stress over
# all pairs = "global", over graph edges = "local").
pred_lists = {
    'nofe': pred_nofe_list,
    'pca': pred_pca_list,
    'tsne': pred_tsne_list,
    'umap': pred_umap_list,
    'phate': pred_phate_list,
    'spatialpca': pred_spatialpca_list,
    'concord': pred_concord_list,
}

def mean_std(arr):
    return f"{np.mean(arr):.4f} ± {np.std(arr):.4f}"

all_scores = {}
for name, preds in pred_lists.items():
    preservation, strs, edge_preservation, edge_strs = [], [], [], []
    strs_sf, edge_strs_sf = [], []
    for i in range(len(graphs)):
        edge_indices = graphs[i].edge_index.detach().cpu().numpy()
        H = graphs[i].normed_values.detach().cpu().numpy()
        L = preds[i]
        preservation.append(distance_preservation(H, L))
        strs.append(stress(H, L))
        edge_preservation.append(edge_distance_preservation(H, L, edge_indices))
        edge_strs.append(edge_stress(H, L, edge_indices))
        strs_sf.append(stress_scale_free(H, L))
        edge_strs_sf.append(edge_stress_scale_free(H, L, edge_indices))
    all_scores[name] = [
        mean_std(preservation), mean_std(strs), mean_std(edge_preservation), mean_std(edge_strs),
        mean_std(strs_sf), mean_std(edge_strs_sf),
    ]

scores_all = pd.DataFrame(
    all_scores,
    index=['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)',
       'stress (global, scale-free)', 'stress (local, scale-free)'],
).T

pd.set_option('display.width', 120)
print(scores_all)
